# Рекурсивный RAG-агент на LangGraph: OpenRouter вместо GigaChat

Копия `demo_recursive_rag_langgraph.ipynb`. Логика графа не менялась — заменён
только клиент модели: вместо SDK `gigachat` используются обычные HTTP-запросы
`requests` к OpenRouter.

## Что настроить перед запуском

| Переменная | Смысл | По умолчанию |
|---|---|---|
| `OPENROUTER_API_KEY` | ключ с https://openrouter.ai/keys | обязательна |
| `OPENROUTER_MODEL` | идентификатор модели | `meta-llama/llama-3.1-8b-instruct` |
| `OPENROUTER_BASE_URL` | базовый URL API | `https://openrouter.ai/api/v1` |

```bash
export OPENROUTER_API_KEY=sk-or-...
export OPENROUTER_MODEL=meta-llama/llama-3.1-8b-instruct
jupyter lab
```

Ключ можно вписать и прямо в ячейку с `OPENROUTER_API_KEY`, но не коммитьте
ноутбук с ключом.

## Что изменилось по сравнению с оригиналом

- `gigachat` / `GigaChat` / `ask_giga` заменены на `requests` и `ask_llm`.
- Список моделей берётся через `GET /models`.
- `ddgs_search` перебирает несколько поисковых движков и скачивает страницы
  через `requests` с таймаутом: один движок часто блокируется.
- `ask_human` не падает при неинтерактивном запуске (например, в nbconvert).
- Ячейки демонстрации поиска не падают при пустой выдаче.
- Выводы ячеек очищены: ноутбук рассчитан на запуск с вашим ключом.


In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import PromptTemplate

from IPython.display import Image, display

from ddgs import DDGS

import requests
import trafilatura

import os

from dotenv import load_dotenv

from typing import List, Dict, Tuple, Literal, Annotated, Any, Optional
from typing_extensions import TypedDict

from operator import add

In [ ]:
load_dotenv('env')

In [ ]:
try:
    with open('./graph.png', 'rb') as fd:
        display(Image(fd.read()))
except FileNotFoundError:
    print("graph.png ещё нет: схема рисуется ниже, в ячейке с draw_mermaid_png().")

In [ ]:
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', '').strip()

if not OPENROUTER_API_KEY:
    print("OPENROUTER_API_KEY не задан.")
    print("Экспортируйте переменную окружения или впишите ключ в эту ячейку.")

In [ ]:
OPENROUTER_BASE_URL = os.environ.get(
    'OPENROUTER_BASE_URL', 'https://openrouter.ai/api/v1'
).rstrip('/')
OPENROUTER_MODEL = os.environ.get(
    'OPENROUTER_MODEL', 'meta-llama/llama-3.1-8b-instruct'
)

print('URL:', OPENROUTER_BASE_URL)
print('Модель:', OPENROUTER_MODEL)

In [ ]:
def list_models():
    """Список моделей OpenRouter обычным GET-запросом."""
    response = requests.get(
        f'{OPENROUTER_BASE_URL}/models',
        headers={'Authorization': f'Bearer {OPENROUTER_API_KEY}'},
        timeout=60,
    )
    response.raise_for_status()
    return [item['id'] for item in response.json()['data']]


models = list_models()
print(f'Доступно моделей: {len(models)}')
print('Первые 10:', models[:10])

In [ ]:
def ask_llm(query: str, model: Optional[str] = None,
            temperature: float = 1.0, max_tokens: int = 1000) -> str:
    """Один запрос к OpenRouter. Без SDK — только requests.post."""
    if not OPENROUTER_API_KEY:
        raise RuntimeError(
            'Не задан OPENROUTER_API_KEY: экспортируйте переменную окружения '
            'или впишите ключ в ячейку выше.'
        )

    payload = {
        'model': model or OPENROUTER_MODEL,
        'messages': [{'role': 'user', 'content': query}],
        'temperature': temperature,
        'max_tokens': max_tokens,
    }
    headers = {
        'Authorization': f'Bearer {OPENROUTER_API_KEY}',
        'Content-Type': 'application/json',
        'HTTP-Referer': 'https://example.local/langgraph-course',
        'X-Title': 'langgraph-recursive-rag',
    }
    response = requests.post(
        f'{OPENROUTER_BASE_URL}/chat/completions',
        headers=headers, json=payload, timeout=120,
    )
    response.raise_for_status()
    return response.json()['choices'][0]['message']['content'].strip()

In [ ]:
ask_llm('Привет, ты кто?', OPENROUTER_MODEL)

In [ ]:
SEARCH_BACKENDS = ('bing', 'duckduckgo', 'yahoo', 'yandex', 'mojeek', 'brave', 'google')


def _ddgs_search(query: str, max_results: int):
    """Перебирает движки: у каждого свои лимиты и блокировки."""
    for backend in SEARCH_BACKENDS:
        try:
            found = DDGS(timeout=8).text(
                query, max_results=max_results, backend=backend
            ) or []
        except Exception:
            continue
        if found:
            return found
    return []


def ddgs_search(query: str, max_results: int = 3):
    results = _ddgs_search(query, max_results)
    for r in results:
        try:
            response = requests.get(
                r['href'], timeout=(5, 20),
                headers={'User-Agent': 'Mozilla/5.0 (compatible; course-bot/1.0)'},
            )
            response.raise_for_status()
            r['doc_html'] = response.text
            r['doc_text'] = trafilatura.extract(response.text) or ''
        except Exception as e:
            print(f"Ошибка при извлечении текста с {r['href']}: {e}")
            r['doc_html'] = ''
            r['doc_text'] = ''
    return results

In [ ]:
res = ddgs_search('как обжаловать штраф?')

In [ ]:
res[0].keys() if res else 'нет результатов поиска'

In [ ]:
res[0]['title'] if res else None

In [ ]:
res[0]['href'] if res else None

In [ ]:
print(res[0]['doc_text'][:2000]) if res else print('нет результатов поиска')

In [ ]:
def call_npa_api(query: str):
    query = query + ' site:consultant.ru/'
    results = ddgs_search(query)
    return results

In [ ]:
def call_court_api(query: str):
    # query = query + ' site:судебныерешения.рф'
    # query = query + ' site:sudrf.ru'
    query = query + ' site:reputation.su'
    results = ddgs_search(query)
    return results

In [ ]:
res = call_court_api('как обжаловать штраф')

In [ ]:
res[0].keys() if res else 'нет результатов поиска'

In [ ]:
res[-1]['href'] if res else None

In [ ]:
def format_docs(search_results: List[Dict]):
    tpl = ''
    for i, r in enumerate(search_results):
        # Используем get с дефолтными значениями на случай, если ключи отсутствуют
        title = r.get('title', '')
        doc_text = r.get('doc_text', '')[:15000] # Ограничиваем длину
        tpl += f"[Source {i}]: {title}\n{doc_text}\n\n" # Добавим пустую строку для разделения
    return tpl

In [ ]:
print(format_docs(res))

In [ ]:
def format_links(search_results):
    links = []
    for r in search_results:
        links.append(f"{r['title']} [{r['href']}\n\n")
    return '\n'.join(links)

In [ ]:
print(format_links(res))

In [ ]:
def ask_human(query: str):
    try:
        return input(query)
    except (EOFError, OSError):
        print(query)
        return 'уточнений нет'

In [ ]:
ask_human('Кто убил Кеннеди?')

In [ ]:
def format_dialog(messages: List[Tuple[str, str]]): # Уточнение типа
    tpl = ''
    for m in messages:
        tpl += f"[{m[0]}]: {m[1]}\n\n" # Добавим перевод строки
    return tpl

In [ ]:
# Промпты
clarification_prompt = PromptTemplate.from_template("""Ты юридический ассистент в Российской Федерации. Тебе дан вопрос пользователя.
Твоя задача определить, надо ли что-то уточнить у пользователя, чтобы выполнить юридический поиск или вопрос пользователя содержит всю необходимую информацию?
Если вопрос содержит всю необходимую информацию ответь: "ок" и больше ничего не пиши.  
Пример: 
Вопрос пользователя: "мне не заплатили по договору, в какой суд подать иск?"
Ответ: Уточите пожалуйста, вы являетесь физическим лицом, индивидуальным предпринимателем или описываете ситуацию компании (юридического лица)?
Вопрос пользователя: "Какой вычет НДФЛ на первого ребенка"
Ответ: ок
[Вопрос пользователя]: "{query}"
Не отвечай на вопрос пользователя, не продолжай диалог, выполни свою задачу.""")

query_concat_prompt = PromptTemplate.from_template("""Ты юридический ассистент. Тебе дан диалог с пользователем, твоя задача собрать этот диалог в один вопрос к справочной правовой системе, который учтет всю информацию, полученную от пользователя.
<dialog>
{dialog}
</dialog>
Не продолжай диалог, только сформулируй один вопрос.
Вопрос к поисковой системе:""")

query_rewrite_prompt = PromptTemplate.from_template("""Ты юридический ассистент. Преобразуй запрос пользователя в краткую, точную поисковую фразу, подходящую для ввода в правовую систему (например, «КонсультантПлюс», «Гарант»).
Требования к результату:
- Сохраняй юридический смысл исходного запроса.
- Используй официальную юридическую терминологию.
- Убирай разговорные формулировки, местоимения и лишние слова.
- Формулируй максимально ёмко — как если бы ты искал в базе законов или судебных решений.
- Не добавляй пояснений, комментариев или примеров.
- Отвечай только одной строкой — самой поисковой фразой.

Примеры преобразований:
Исходный запрос: "Что будет, если не платить алименты?"
Поисковая фраза: Уклонение от уплаты алиментов ответственность
Исходный запрос: "Могу ли я вернуть товар, если он не подошёл по цвету?"
Поисковая фраза: Возврат товара ненадлежащего качества цвет
Исходный запрос: "Какие льготы у ветеранов труда в Москве?"
Поисковая фраза: Льготы ветеранам труда Москва
Исходный запрос: "Судебная практика по увольнению за прогул без объяснений"
Поисковая фраза: Судебная практика увольнение за прогул отсутствие объяснений
Исходный запрос: "Что говорит 217-ФЗ о добровольной ликвидации ООО?"
Поисковая фраза: ФЗ-217 добровольная ликвидация ООО

Теперь обработай следующий запрос (не отвечая на сам вопрос, только сделай перефразировку):
[Исходный запрос]: "{query}"
Поисковая фраза:""")

classification_prompt = PromptTemplate.from_template("""Ты юридический ассистент. Проанализируй следующий запрос пользователя и определи, к какой категории он относится:
- "НПА" — если запрос касается нормативно-правовых актов: законов, кодексов, статей, постановлений, правил, положений, требований законодательства, юридических норм, обязанностей по закону и т.п.
- "Судебное" — если запрос касается судебных решений, прецедентов, примеров из практики судов, разборов дел, аргументов в суде, толкования норм судами, позиции ВС/Арбитражных судов и т.п.
Важно: отвечай **только одним словом** — либо "НПА", либо "Судебное".
Запрос: "{query}"
Ответ (одно слово):""")

rag_prompt = PromptTemplate.from_template("""Ты — опытный юрист. На основе представленных документов ответь на вопрос пользователя, строго придерживаясь следующей структуры:
1. **Законодательство**
   Перечисли применимые нормативно-правовые акты, статьи, положения. Укажи, что именно они устанавливают по вопросу. Ссылайся только на документы, приведённые ниже. Не добавляй внешние знания.
2. **Судебная практика**
   Приведи примеры из судебных решений, если они есть в документах. Укажи: какой суд, по какому делу, какую позицию занял. Если практики нет — напиши: "В представленных документах отсутствует судебная практика по данному вопросу."
3. **Вывод**  
   Дай краткий, однозначный ответ на вопрос, основанный на совокупности НПА и судебной практики. Формулируй как юридическую позицию: что допускается, запрещено, рекомендовано, возможно в суде и т.п.
Требования:
- Отвечай только на основе документов.
- Не вымышляй нормы или дела.
- Используй официальную юридическую терминологию.
- Будь точен, лаконичен, структурирован.
[Вопрос]: "{query}"
[Документы]:
{docs}
Ответ:""")

reflection_prompt = PromptTemplate.from_template("""Ты — эксперт-юрист. Проанализируй вопрос пользователя и черновик ответа. 
Оцени, насколько черновик ответа полностью раскрывает вопрос с точки зрения:
- нормативно-правового регулирования (НПА),
- судебной практики (если уместно),
- возможных нюансов, условий, исключений.
Если ответ:
- полностью раскрывает вопрос по всем значимым аспектам → верни одну фразу: "ок" и больше ничего не пиши.
- недостаточен, содержит пробелы или требует уточнения → сформулируй один точный поисковой запрос для справочно-правовой системы (например, «КонсультантПлюс», «Гарант»), который поможет дополнить ответ. Запрос должен быть кратким, ёмким, с использованием юридической терминологии.
Не добавляй пояснений. Отвечай только "ок" или поисковой фразой.
---
Пример 1:
Вопрос: Можно ли уволить сотрудника за однократный прогул без предупреждений?
Черновик ответа: Да, можно. Согласно ст. 81 ТК РФ, прогул является основанием для увольнения.
Твой ответ: Судебная практика увольнение за прогул отсутствие уведомления
Пример 2:
Вопрос: Облагается ли НДС продажа доли в уставном капитале ООО?
Черновик ответа: Нет, передача доли в уставном капитале ООО не облагается НДС на основании п. 2 ст. 149 НК РФ.
Твой ответ: ок
---
[Вопрос]: "{query}"
[Черновик ответа]: {response}
[Твой ответ]:""")

final_answer_prompt = PromptTemplate.from_template("""Ты — опытный юрист. На основе представленных документов ответь на вопрос пользователя, строго придерживаясь следующей структуры:
1. **Законодательство**
   Перечисли применимые нормативно-правовые акты, статьи, положения. Укажи, что именно они устанавливают по вопросу. Ссылайся только на документы, приведённые ниже. Не добавляй внешние знания.
2. **Судебная практика**
   Приведи примеры из судебных решений, если они есть в документах. Укажи: какой суд, по какому делу, какую позицию занял. Если практики нет — напиши: "В представленных документах отсутствует судебная практика по данному вопросу."
3. **Вывод**  
    Дай развернутый ответ на основе всех предложенных документов. Формулируй как юридическую позицию: что допускается, запрещено, рекомендовано, возможно в суде и т.п.

Требования:
- Отвечай только на основе документов.
- Не вымышляй нормы или дела.
- Используй официальную юридическую терминологию.
- Будь точен, лаконичен, структурирован.
[Вопрос]: "{query}"
[Документы]:
{docs}
Ответ:""")

In [ ]:
class MyState(TypedDict):
    # Используем Optional для полей, которые могут отсутствовать в начале
    query: str  # Исходный запрос пользователя
    search_query: Optional[str] # Текущий поисковый запрос агента
    messages: Annotated[List[Tuple[str, str]], add]  # Сообщения
    clarification: Optional[str]
    need_clarify_question: Optional[bool]
    category: Optional[str]
    docs: Annotated[List[Dict], add] # Документы, полученные в результате поиска
    answers: Annotated[List[Any], add] # Промежуточные ответы ассистента
    final_answer: str
    need_re_search: Optional[bool]
    re_search_cnt: Optional[int]
    clarification_cnt: Optional[int]
    verbose: Optional[bool]

In [ ]:
# Nodes
def setup_node(state: MyState) -> MyState:
    state_update = dict()
    state_update['search_query'] = state['query']
    state_update['messages'] = [('user', state['query'])]
    state_update['clarification'] = None
    state_update['need_clarify_question'] = None
    state_update['category'] = None
    state_update['docs'] = []
    state_update['answers'] = []
    state_update['final_answer'] = None
    state_update['need_re_search'] = None
    state_update['re_search_cnt'] = 0
    state_update['clarification_cnt'] = 0
    state_update['verbose'] = state.get('verbose', False)
    
    return state_update
    
def clarify_node(state: MyState) -> MyState:
    """
    Проверяем нуждается ли вопрос в уточнении от пользователя.
    Если нуждается, то формулируем уточняющий вопрос
    """
    query = state["search_query"]
    prompt = clarification_prompt.format(query=query)
    gen = ask_llm(prompt, OPENROUTER_MODEL)

    state_update = dict()
    state_update["need_clarify_question"] = False if (len(gen) < 10 and 'ок' in gen.lower()) else True
    state_update['messages'] = [('tool_clarify', gen)]
    state_update['clarification'] = gen
    state_update['clarification_cnt'] = state['clarification_cnt'] + 1
    
    if state['verbose']:
        print("clarify_node:", gen)
    
    return state_update


def human_clarify_node(state: MyState) -> MyState:
    """Просим пользователя ответить на уточняющий вопрос"""
    value = ask_human(state['clarification'])
    
    message = ('user', value)
    
    state_update = dict()
    state_update['messages'] = [message]

    if state['verbose']:
        print("human_clarify_node:", value)
    
    return state_update

    
def query_concat_node(state: MyState) -> MyState:
    """Объединяем диалог с пользователем в один вопрос для поиска"""
    dialog = format_dialog(state['messages'])
    prompt = query_concat_prompt.format(dialog=dialog)
    gen = ask_llm(prompt, OPENROUTER_MODEL)

    message = ('tool_concat', gen)
    
    state_update = dict()
    state_update['search_query'] = gen
    state_update['messages'] = [message]

    if state['verbose']:
        print("query_concat_node:", gen)
    
    return state_update

    
def rewrite_node(state: MyState) -> MyState:
    query = state["search_query"]
    prompt = query_rewrite_prompt.format(query=query)
    rewritten = ask_llm(prompt, OPENROUTER_MODEL)

    message = ('tool_rewrite', rewritten)
    
    state_update = dict()
    state_update["search_query"] = rewritten
    state_update['messages'] = [message]

    if state['verbose']:
        print("rewrite_node:", message)
        
    return state_update


def classify_node(state: MyState) -> MyState:
    """Классифицируем запрос по типу поиск - в правовой системе или в судебной практике"""
    query = state["search_query"]
    prompt = classification_prompt.format(query=query)
    category = ask_llm(prompt, OPENROUTER_MODEL)

    message = ('tool_classify', category)
    
    state_update = dict()
    state_update["category"] = category
    state_update['messages'] = [message]
    state_update['docs'] = []
    
    if state['verbose']:
        print("classify_node:", category)
    
    return state_update


def search_npa_node(state: MyState) -> MyState:
    results = call_npa_api(state["search_query"])

    message_text = format_links(results)
    message = ('result_search_npa', message_text)
    
    state_update = dict()
    state_update["docs"] = results
    state_update['messages'] = [message]

    if state['verbose']:
        print("search_npa_node:", message_text)
    
    return state_update


def search_court_node(state: MyState) -> MyState:
    results = call_court_api(state["search_query"])

    message_text = format_links(results)
    message = ('result_search_court', message_text)
    
    state_update = dict()
    state_update["docs"] = results
    state_update['messages'] = [message]

    if state['verbose']:
        print("search_court_node:", message_text)
    
    return state_update


def answer_node(state: MyState) -> MyState:
    query=state["search_query"]
    # Проверка на существование docs
    docs = state.get("docs", [])
    if not docs:
        # Обработка случая, если документы не найдены
        answer = "Извините, по вашему запросу не удалось найти подходящие документы."
    else:
        prompt = rag_prompt.format(query=query, docs=format_docs(docs))
        answer = ask_llm(prompt, OPENROUTER_MODEL)
    
    answer_data = {
        'title': query,
        'doc_text': answer
    }
    
    message = ('tool_rag', answer)

    state_update = dict()
    state_update["answers"] = [answer_data]
    state_update['messages'] = [message]
    
    if state['verbose']:
        print("answer_node:", answer)
    
    return state_update


def reflect_node(state: MyState) -> MyState:
    answer = state['answers'][-1]
    prompt = reflection_prompt.format(query=state["search_query"], response=answer)
    gen = ask_llm(prompt, OPENROUTER_MODEL)

    message = ('tool_reflect', gen)

    state_update = dict()
    state_update["need_re_search"] = False if (len(gen) < 10 and 'ок' in gen.lower()) else True
    if state_update["need_re_search"]:
        state_update["search_query"] = gen
    
    state_update['messages'] = [message]
    state_update['re_search_cnt'] = state['re_search_cnt'] + 1

    if state['verbose']:
        print("reflect_node:", gen)
    
    return state_update


def final_answer_node(state: MyState) -> MyState:
    query=state["query"]
    # Проверка на существование docs
    docs = state["answers"]
    if not docs:
        # Обработка случая, если документы не найдены
        answer = "Извините, по вашему запросу не удалось найти подходящие документы."
    elif len(docs) == 1:
        answer = docs[0]["doc_text"]
    else:
        prompt = final_answer_prompt.format(query=query, docs=format_docs(docs))
        answer = ask_llm(prompt, OPENROUTER_MODEL)

    message = ('tool_final_answer', answer)
    
    state_update = dict()
    state_update['final_answer'] = answer
    state_update['messages'] = [message]

    if state['verbose']:
        print("answer_node:", answer)
    
    return state_update
    

In [ ]:
# Decisions
def check_need_human(state: MyState) -> Literal['human_clarify_node', 'rewrite_node']:
    clarification_cnt = state['clarification_cnt']
    need_clarify_question = state['need_clarify_question']

    if clarification_cnt > 1:
        return 'rewrite_node'
    if need_clarify_question:
        return 'human_clarify_node'
    return 'rewrite_node'


def check_search_type(state: MyState) -> Literal['search_npa_node', 'search_court_node']:
    # Проверка существования ключа
    category = state['category'].lower()
    if 'нпа' in category:
        return 'search_npa_node'
    if 'суд' in category:
        return 'search_court_node'
    return 'search_npa_node'  # По умолчанию ищем в НПА


def check_need_re_search(state: MyState) -> Literal['classify_node', 'final_answer_node']:
    # Проверка существования ключей
    re_search_cnt = state['re_search_cnt']
    need_re_search_flag = state['need_re_search']

    if re_search_cnt > 1:
        return 'final_answer_node'
    if need_re_search_flag:
        return 'classify_node' # После рефлексии снова классифицируем вопрос и идем в поиск
    return 'final_answer_node'

In [ ]:
# Workflow definition
workflow = StateGraph(MyState)
workflow.add_node("setup_node", setup_node)
workflow.add_node("clarify_node", clarify_node)
workflow.add_node("human_clarify_node", human_clarify_node)
workflow.add_node("query_concat_node", query_concat_node)
workflow.add_node("rewrite_node", rewrite_node)
workflow.add_node("classify_node", classify_node)
workflow.add_node("search_npa_node", search_npa_node)
workflow.add_node("search_court_node", search_court_node)
workflow.add_node("answer_node", answer_node)
workflow.add_node("reflect_node", reflect_node)
workflow.add_node("final_answer_node", final_answer_node)


workflow.add_edge(START, "setup_node")
workflow.add_edge("setup_node", "clarify_node")
workflow.add_conditional_edges("clarify_node", check_need_human)
workflow.add_edge("human_clarify_node", "query_concat_node")
workflow.add_edge("query_concat_node", "rewrite_node")
workflow.add_edge("rewrite_node", "classify_node")
workflow.add_conditional_edges("classify_node", check_search_type)
workflow.add_edge("search_npa_node", "answer_node")
workflow.add_edge("search_court_node", "answer_node")
workflow.add_edge("answer_node", "reflect_node")
workflow.add_conditional_edges("reflect_node", check_need_re_search)
workflow.add_edge("final_answer_node", END)

# Компиляция графа
graph = workflow.compile()

In [ ]:
graph

In [ ]:
try:
    png = graph.get_graph().draw_mermaid_png()
    with open('./graph.png', 'wb') as fd:
        fd.write(png)
    display(Image(png))
except Exception as exc:
    print('Не удалось отрисовать PNG:', type(exc).__name__)
    print(graph.get_graph().draw_mermaid())

In [ ]:
state = dict()
state['query'] = 'Как обжаловать расторжение договора аренды?'

# stream(from_checkpoint=None) означает запуск с самого начала
# stream_mode="values" тоже может быть полезен
for step_result in graph.stream(state, stream_mode="updates"):
    print("--- ШАГ ---")
    # step_result - это словарь, где ключ - это имя узла или "__end__", а значение - обновленное состояние (state)
    for node_name, updated_state in step_result.items():
        print(f"Выполнен узел: {node_name}")
        # Здесь вы можете инспектировать updated_state
        print(f"  Текущий search_query: {updated_state.get('search_query', 'N/A')}")
        print(f"  Текущая category: {updated_state.get('category', 'N/A')}")
        print(f"  Текущий answer: {updated_state.get('answer', 'N/A')[:100]}...") # Первые 100 символов
        print(f"  need_clarify_question: {updated_state.get('need_clarify_question', 'N/A')}")
        print(f"  need_re_search: {updated_state.get('need_re_search', 'N/A')}")
        print(f"  clarification_cnt: {updated_state.get('clarification_cnt', 'N/A')}")
        print(f"  re_search_cnt: {updated_state.get('re_search_cnt', 'N/A')}")
        # Печать последних нескольких сообщений для понимания диалога
        messages = updated_state.get('messages', [])
        if messages:
            print("  Последние сообщения:")
            for msg in messages[-3:]: # Печатаем последние 3 сообщения
                print(f"    {msg}")
        print("-" * 20)

In [ ]:
print(updated_state['final_answer'])

In [ ]:
state = dict()

# state['query'] = 'Какие есть ограничения на увольнение беременной сотрудницы?'
state['query'] = 'Как обжаловать расторжение договора аренды?'
state['verbose'] = True

In [ ]:
res = graph.invoke(state)

In [ ]:
print(res['final_answer'])

In [ ]:
len(res['messages'])

In [ ]:
print(format_dialog(res['messages']))

In [ ]:
res.keys()

In [ ]:
res['clarification_cnt']

In [ ]:
res['re_search_cnt']

In [ ]:
res['need_re_search']

In [ ]:
res['search_query']

In [ ]:
res['clarification']

In [ ]:
res['need_clarify_question']

In [ ]:
res['category']